In [0]:
#Setup
from pyspark.sql import functions as F
from pyspark.sql.window import Window

BASE = "/Volumes/gr5069/raw/f1_data"

pit_stops             = spark.read.csv(f"{BASE}/pit_stops.csv",             header=True, inferSchema=True)
races                 = spark.read.csv(f"{BASE}/races.csv",                 header=True, inferSchema=True)
results               = spark.read.csv(f"{BASE}/results.csv",               header=True, inferSchema=True)
drivers               = spark.read.csv(f"{BASE}/drivers.csv",               header=True, inferSchema=True)
constructors          = spark.read.csv(f"{BASE}/constructors.csv",          header=True, inferSchema=True)
constructor_standings = spark.read.csv(f"{BASE}/constructor_standings.csv", header=True, inferSchema=True)

# 1. [10 pts] What was the average time each driver spent at the pit stop for each race? Provide also the slowest and fastest pit stop in each race.

## 1.1 What was the average time each driver spent at the pit stop for each race?

**Logic:**  

The `pit_stops` table has one row per individual pit stop event, identified by `raceId`, `driverId`, and `stop` (which stop number it was for that driver in that race). The `duration` column records how long that single stop took in seconds. 

However, the data contains some anomalous row where `duration` is stored as a `'MM:SS.mmm'` string (e.g. `'16:44.718'`, meaning 16 minutes 44.718 seconds) instead of a plain decimal number. This happens because that particular pit stop was extremely long — likely due to a mechanical issue during the stop.

To handle this, I first create a new column `duration_seconds` in dataframe `pit_stops_clean` that:
- Detects whether `duration` contains a colon (`:`), indicating the `MM:SS` format.
- If yes, splits on `:`, multiplies the minutes part by 60, and adds the seconds part to convert to total seconds.
- If no, simply casts the value directly to `double`.

I then `groupBy('driverId', 'raceId')` and compute `avg('duration_seconds')` to get each driver's average pit stop time per race, ordered by `driverId` then `raceId` so it is easy to follow each driver's history across races.
   
 

In [0]:
pit_stops_clean = pit_stops.withColumn(
    "duration_seconds",
    F.when(
        F.col("duration").contains(":"),
        F.split(F.col("duration"), ":")[0].cast("double") * 60 +
        F.split(F.col("duration"), ":")[1].cast("double")
    ).otherwise(
        F.col("duration").cast("double")
    )
)

driver_pit_stats = (
    pit_stops_clean
    .groupBy("driverId","raceId" )
    .agg(
        F.round(F.avg("duration_seconds"), 3).alias("avg_pit_duration_s")
    )
    .orderBy("driverId", "raceId")
)

display(driver_pit_stats)

**Code Explanation:**  
- `F.col('duration').contains(':')` checks whether the duration value is in `MM:SS.mmm` format. This is the condition inside `F.when()`.
- `F.split(F.col('duration'), ':')[0].cast('double') * 60` extracts the minutes part (index 0 after splitting on `:`), casts it to a number, and converts to seconds by multiplying by 60.
- `F.split(F.col('duration'), ':')[1].cast('double')` extracts the seconds-and-milliseconds part (index 1) and adds it to the minutes-in-seconds above.
- `.otherwise(F.col('duration').cast('double'))` handles all normal rows where duration is already a plain decimal like `23.418` — it just casts directly to double.
- `.groupBy('driverId', 'raceId').agg(F.round(F.avg('duration_seconds'), 3))` groups every pit stop event by driver and race, then computes the mean duration, rounded to 3 decimal places (millisecond precision).
- `.orderBy('driverId', 'raceId')` sorts the output so all races for a given driver appear together in chronological order, making it easy to read.

## 1.2 Provide also the slowest and fastest pit stop in each race.
**Logic:**  
The question asks for the single slowest and fastest pit stop *within each race* — not per driver. This means I only need to `groupBy('raceId')` (not `driverId`) and apply `min()` and `max()` on `duration_seconds`. `min()` gives the fastest (shortest) stop and `max()` gives the slowest (longest) stop recorded in that race across all drivers and all of their stops.

I use the same `pit_stops_clean` DataFrame (with the corrected `duration_seconds` column) so the anomalous `'16:44.718'` row is correctly converted to `1004.718` seconds and included as a legitimate — if extreme — data point rather than being dropped.


In [0]:
race_pit_extremes = (
    pit_stops_clean
    .groupBy("raceId")
    .agg(
        F.round(F.min("duration_seconds"), 3).alias("race_fastest_pit_s"),
        F.round(F.max("duration_seconds"), 3).alias("race_slowest_pit_s"),
    )
)
display(race_pit_extremes)

**Code Explanation:**

- `pit_stops_clean` — we use the cleaned DataFrame where the anomalous duration value ('16:44.718') has already been converted to 1004.718 seconds, ensuring all values in duration_seconds are numeric and can be aggregated without errors.
- `.groupBy("raceId")` — groups all pit stop records by race. Notice that driverId is intentionally not included here — we want the extreme values across all drivers and all stops within each race, not per individual driver.
- `F.min("duration_seconds").alias("race_fastest_pit_s")` — finds the single shortest pit stop duration recorded in that race across all drivers and all their stops. A smaller duration means a faster pit stop, so min() gives us the fastest.
- `F.max("duration_seconds").alias("race_slowest_pit_s")` — symmetrically, finds the longest single pit stop in the race. A larger duration means a slower stop, so max() gives us the slowest.
- `F.round(..., 3`) — rounds the result to 3 decimal places (millisecond precision), preventing unnecessary floating point noise in the output.
- `.alias(...) `— renames the aggregated columns to descriptive names so the output table is self-explanatory when displayed.

### **Alternative routes to answer 1.2**
**Logic:**  
Instead of a second `groupBy`, I use a **window function** partitioned by `raceId` to compute `min` and `max` of `milliseconds` *inline*, keeping the full row detail (driverId, stop number, lap number). This lets me identify not just *what* the fastest/slowest duration was, but *which driver, on which stop, on which lap* it occurred — information that a plain `groupBy` aggregation would discard.

In [0]:
from pyspark.sql.window import Window

pit_stops_extremes = (
    pit_stops
    .withColumn("race_fastest_ms", F.min("milliseconds").over(Window.partitionBy("raceId")))
    .withColumn("race_slowest_ms", F.max("milliseconds").over(Window.partitionBy("raceId")))
    .filter(
        (F.col("milliseconds") == F.col("race_fastest_ms")) |
        (F.col("milliseconds") == F.col("race_slowest_ms"))
    )
    .withColumn("type", 
        F.when(F.col("milliseconds") == F.col("race_fastest_ms"), "fastest")
         .otherwise("slowest")
    )
    .select("raceId", "driverId", "stop", "lap", "duration", "milliseconds", "type")
    .orderBy("raceId", "type")
)

display(pit_stops_extremes)

**Code Explanation:**  
- `Window.partitionBy('raceId')` defines a window scoped to one race at a time, with no ordering needed since we want global min/max within the partition.
- `F.min('milliseconds').over(race_win)` adds a new column to every row containing the race-wide minimum pit duration. Same for `max`.
- `.filter(...)` then keeps only the rows where the driver's `milliseconds` equals the race minimum or maximum — i.e. the actual fastest/slowest stop events.
- `F.when(...).otherwise(...)` labels each surviving row as `'fastest'` or `'slowest'` for readability.
- This approach is more informative than `groupBy` because it preserves the `stop`, `lap`, and `driverId` context of the extreme events.

# 2. [20 pts] Rank order by finishing position the average time spent at the pit stop in each race.

**Logic:**
1. Join the per-driver pit-stop averages (from Q1) with `results` to get the actual `positionOrder` (finishing position) for each driver in each race.
2. Use a **window function** partitioned by `raceId` to **rank** drivers within each race by their average pit-stop duration (ascending = fastest first).
3. Display the finishing position alongside the pit-stop rank so the relationship is visible.

In [0]:
pit_vs_finish = (
    driver_pit_stats
    .join(
        results.select("raceId", "driverId", "positionOrder"),
        on=["raceId", "driverId"],
        how="inner",
    )
)
display(pit_vs_finish)

In [0]:
race_window = Window.partitionBy("raceId").orderBy(F.asc("avg_pit_duration_s"))

q2_result = (
    pit_vs_finish
    .withColumn("pit_speed_rank", F.rank().over(race_window))
    .select(
        "raceId",
        "driverId",
        "avg_pit_duration_s",
        "pit_speed_rank",
        "positionOrder",
    )
    .orderBy("raceId", "pit_speed_rank")
)

display(q2_result)

**Code Explanation:**

**Part 1 — `pit_vs_finish`**

- `driver_pit_stats` — the DataFrame from Q1 containing each driver's average pit stop duration (`avg_pit_duration_s`) per race, grouped by `raceId` and `driverId`.

- `.join(results.select("raceId", "driverId", "positionOrder"), on=["raceId", "driverId"], how="inner")` — joins with the `results` table to bring in each driver's finishing position (`positionOrder`). I join on **both** `raceId` and `driverId` simultaneously to correctly match each driver's pit stats to their result in the **same** race. `how="inner"` means only rows that exist in **both** DataFrames are kept — drivers who have pit stop data but no recorded finishing position (or vice versa) are excluded.

- `results.select("raceId", "driverId", "positionOrder")` — I only select the three columns I actually need from `results`, avoiding unnecessary columns and potential name conflicts.

**Part 2 — `q2_result`**

- `race_window = Window.partitionBy("raceId").orderBy(F.asc("avg_pit_duration_s"))` — defines the ranking rule: partition the data by race (so ranking is done separately within each race), then order by average pit duration ascending (shortest = rank 1).

- `.withColumn("pit_speed_rank", F.rank().over(race_window))` — applies the ranking rule to create a new column `pit_speed_rank`. Rank 1 = fastest average pit stop in that race. If two drivers tie on `avg_pit_duration_s`, they receive the same rank and the next rank is skipped (e.g. 1, 1, 3).

- `.select(...)` — keeps only the five relevant columns for the final output, dropping any extra columns produced during the join.

- `.orderBy("raceId", "pit_speed_rank")` — sorts the output first by race, then by pit speed rank within each race, so it is easy to read each race's pit stop leaderboard from fastest to slowest.

# 3. [20 pts] Insert the missing code (e.g: ALO for Alonso) for drivers based on the 'drivers' dataset. Explain your logic.

**Logic:**

The `drivers` table has a `code` column (a 3-letter abbreviation like `ALO` for Alonso, `HAM` for Hamilton) but many rows have `null` in this field — particularly for older or less prominent drivers.

My imputation strategy follows the real-world FIA convention: a driver's code is typically the **first 3 letters of their surname, uppercased**.

**Steps:**
1. **Derive a candidate code** for every driver based on their surname, following the FIA convention of using the first 3 letters uppercased. However, some drivers have compound surnames with a prefix (e.g. "de Cesaris", "de la Rosa"). Taking the first 3 characters of these surnames would only capture the prefix (e.g. "DE"), which is not meaningful. Therefore, 
    - I first check whether the surname contains a space — if it does, I split on the space and take the last word of the surname (e.g. "Cesaris" from "de Cesaris"), then take its first 3 characters uppercased (e.g. "CES"). 
    - For simple single-word surnames, I take the first 3 characters directly (e.g. "Hamilton" → "HAM").
2. **Prefer the existing code** where it is already present and non-empty. Only fall back to the derived candidate for rows where code is \N.

In [0]:
# Step 1 – Derive a candidate code from surname (first 3 chars, uppercased)
drivers_with_candidate = drivers.withColumn(
    "candidate_code",
    F.when(
        F.col("surname").contains(" "),
        F.upper(F.substring(F.element_at(F.split(F.col("surname"), " "), -1), 1, 3))
    ).otherwise(
        F.upper(F.substring(F.col("surname"), 1, 3))
    )
)
display(drivers_with_candidate)

In [0]:
# Step 2 – Use existing code where available; fall back to candidate
drivers_filled = drivers_with_candidate.withColumn(
    "code_filled",
    F.when(
        F.col("code") != "\\N",
        F.col("code"),
    ).otherwise(F.col("candidate_code")),
)
display(drivers_filled)

**Code Explanation:**

**Step 1 — Derive candidate code**

- `F.col("surname").contains(" ")` — checks whether the surname contains a space, which indicates a compound surname with a prefix such as `"de Cesaris"` or `"de la Rosa"`. This is the condition that determines which derivation rule to apply.

- `F.split(F.col("surname"), " ")` — splits the surname into a list of words by space. For example, `"de la Rosa"` becomes `["de", "la", "Rosa"]`.

- `F.element_at(..., -1)` — extracts the **last element** of the list (index -1 = last). This gives us the meaningful part of the surname, ignoring the prefix — `"Rosa"` from `"de la Rosa"`, `"Cesaris"` from `"de Cesaris"`.

- `F.substring(..., 1, 3)` — takes the first 3 characters starting at position 1 (1-indexed in Spark). Applied to `"Cesaris"` gives `"Ces"`.

- `F.upper(...)` — uppercases the result to match the FIA format convention, giving the final candidate code e.g. `"CES"`.

- `.otherwise(F.upper(F.substring(F.col("surname"), 1, 3)))` — for simple single-word surnames with no space, just take the first 3 characters directly and uppercase them, e.g. `"Hamilton"` → `"HAM"`.

**Step 2 — Fill missing codes**

- `F.when(F.col("code") != "\\N", F.col("code"))` — checks if the existing code is not `\N`. In this dataset, missing values are stored as the string `"\N"` rather than a true null, so we check for this specific string. If the code is a real value (not `\N`), we keep it as-is.

- `.otherwise(F.col("candidate_code"))` — for rows where `code` is `\N` (i.e. missing), we fall back to the `candidate_code` derived in Step 1.

- `.alias("code_filled")` — stores the result in a new column `code_filled`, preserving the original `code` column so we can later audit which values were imputed and which were already present.

# 4. [20 pts] Who is the youngest and the oldest driver in each race? Create a new column called “Age”. Explain your definition of "age".

**Logic:**

**Definition of Age:** A driver's age is defined as the number of **complete years** elapsed between their date of birth (`dob`) and the **date of the race** (`races.date`). 

**Steps:**
1. Join `results` with `races` (to get the race date) and `drivers` (to get `dob`), using `raceId` and `driverId` as join keys respectively.
2. Compute `age` = `floor(datediff(race_date, dob) / 365)`. `datediff` gives the number of calendar days between the two dates; dividing by 365.25 (the mean Gregorian year, accounting for leap years) converts to years; `floor` truncates to whole completed years.
3. Use a window function `partitionBy('raceId').orderBy('age')` with `rank()` to identify the youngest and oldest driver in each race. For the youngest, rank by age ascending (rank 1 = smallest age); for the oldest, rank by age descending (rank 1 = largest age). Filter to `rank == 1` to extract only the youngest and oldest driver per race into two separate DataFrames.
4. Join both DataFrames onto the `races` table so that each race appears as a single row with the youngest driver's information on one side and the oldest driver's information on the other.

In [0]:
# Step 1 – Join results with race date and driver dob
race_drivers = (
    results
    .select("raceId", "driverId")
    .join(races.select("raceId", "name", "year", "date"), on="raceId", how="inner")
    .join(
        drivers.select("driverId", "forename", "surname", "dob"),
        on="driverId",
        how="inner",
    )
)
display(race_drivers)

In [0]:
# Step 2 – Compute age in complete years at the time of the race
race_drivers_age = race_drivers.withColumn(
    "age",
    F.floor(F.datediff(F.col("date"), F.col("dob")) / 365).cast("int"),
)
display(race_drivers_age)

In [0]:
# Step 3 – Find youngest and oldest per race
youngest = (
    race_drivers_tagged
    .filter(F.col("youngest_in_race") == True)
    .select(
        "raceId",
        F.concat_ws(" ", "forename", "surname").alias("youngest_driver"),
        F.col("age").alias("youngest_age"),
        F.col("dob").alias("youngest_dob"),
    )
)

oldest = (
    race_drivers_tagged
    .filter(F.col("oldest_in_race") == True)
    .select(
        "raceId",
        F.concat_ws(" ", "forename", "surname").alias("oldest_driver"),
        F.col("age").alias("oldest_age"),
        F.col("dob").alias("oldest_dob"),
    )
)

In [0]:
# Step 4 - Reshape to one row per race with youngest and oldest side by side
q4_result = (
    races.select("raceId", "name", "date", "year")
    .join(youngest, on="raceId", how="left")
    .join(oldest, on="raceId", how="left")
    .select(
        "raceId", "name", "date", "year",
        "youngest_driver", "youngest_age", "youngest_dob",
        "oldest_driver", "oldest_age", "oldest_dob",
    )
    .orderBy("year", "name")
)

display(q4_result)

**Code Explanation:**

**Step 1 — Join tables**

- `results.select("raceId", "driverId")` — selects only the two ID columns we need from `results`, which tells us which drivers participated in which races.
- `.join(races.select("raceId", "name", "year", "date"), on="raceId", how="inner")` — brings in the race date, name, and year. The race `date` is essential for computing each driver's age at the time of the race.
- `.join(drivers.select("driverId", "forename", "surname", "dob"), on="driverId", how="inner")` — brings in each driver's date of birth and name. `how="inner"` keeps only rows with matching records in all three tables.

**Step 2 — Compute age at race time**

- `F.datediff(F.col("date"), F.col("dob"))` — returns the number of calendar days between the race date and the driver's date of birth.
- `/ 365.25` — converts days to years using 365.25 to correctly account for leap years.
- `F.floor(...).cast("int")` — truncates to whole completed years, matching the standard real-world definition of age.

**Step 3 — Find youngest and oldest per race**

- `Window.partitionBy("raceId").orderBy("age")` — for `youngest`: scopes the window to one race and orders by age ascending, so rank 1 = the youngest driver in that race.
- `Window.partitionBy("raceId").orderBy(F.desc("age"))` — for `oldest`: same but orders by age descending, so rank 1 = the oldest driver in that race.
- `F.rank().over(...)` — assigns rank 1 to the driver with the minimum/maximum age. `rank()` handles ties correctly — if two drivers share the same age, both get rank 1.
- `.filter(F.col("rank") == 1)` — keeps only the youngest/oldest driver(s) per race.
- `F.concat_ws(" ", "forename", "surname")` — combines first and last name into a single readable column.

**Step 4 — Reshape to one row per race**

- `races.select(...).join(youngest, ...).join(oldest, ...)` — joins both DataFrames onto the `races` table using `raceId`. `how="left"` ensures all races are retained even if no driver data is found.
- The final `.select(...)` arranges columns in a clean side-by-side layout — race info on the left, youngest driver in the middle, oldest driver on the right.
- `.orderBy("year", "name")` sorts chronologically by season and alphabetically by race name within each season.

# 5. [20 pts] At any given race, how many podiums does each driver have? create three new columns to show - on any given race - the number of wins, the number of 2nd places, and the number of 3rd places for each driver


**Logic:**
The question asks: at the moment of *any given race*, how many total wins, 2nd places, and 3rd places has each driver accumulated **up to and including that race**?

**Steps:**
1. Create binary (1/0) indicator columns on `results`: `is_win` = 1 when `positionOrder == 1`, `is_p2` = 1 when `positionOrder == 2`, `is_p3` = 1 when `positionOrder == 3`, and 0 otherwise. These are applied to **all** race results (not just podium finishes) so that non-podium races contribute 0 to the running totals.
2. Define a **cumulative window** per driver: `partitionBy('driverId').orderBy('raceId').rowsBetween(Window.unboundedPreceding, Window.currentRow)`. This window, for each row (race), includes all previous races by the same driver plus the current one.
3. Apply `F.sum('is_win').over(cum_win)` (and similarly for `is_p2`, `is_p3`) to compute running totals of each podium type.
4. Join with `races` and `drivers` for readable names and order by year, race, and total podiums descending.

In [0]:
# Step 1 – Indicator columns for each podium position
results_with_flags = (
    results
    .withColumn("is_win", F.when(F.col("positionOrder") == 1, 1).otherwise(0))
    .withColumn("is_p2",  F.when(F.col("positionOrder") == 2, 1).otherwise(0))
    .withColumn("is_p3",  F.when(F.col("positionOrder") == 3, 1).otherwise(0))
)
display(results_with_flags)

In [0]:
# Step 2 – Cumulative window per driver ordered by raceId (proxy for chronology)
cum_win = (
    Window
    .partitionBy("driverId")
    .orderBy("raceId")
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)
)
# Step 3 & 4 — Compute cumulative counts and format output
q5_result = (
    all_results_flags
    .withColumn("cum_wins", F.sum("is_win").over(cum_win))
    .withColumn("cum_p2",   F.sum("is_p2").over(cum_win))
    .withColumn("cum_p3",   F.sum("is_p3").over(cum_win))
    .join(races.select("raceId", "name", "year"), on="raceId", how="left")
    .join(
        drivers.select("driverId", "forename", "surname"),
        on="driverId",
        how="left",
    )
    .select(
        "year", "name",
        F.concat_ws(" ", "forename", "surname").alias("driver"),
        "positionOrder",
        "cum_wins", "cum_p2", "cum_p3"
    )
    .orderBy("year", "name", "positionOrder")
)

display(q5_result)

**Code Explanation:**

**Step 1 — Create podium indicator columns**

- `F.when(F.col("positionOrder") == 1, 1).otherwise(0)` — creates a binary (1/0) flag for each podium position. This converts a categorical finishing position into a summable numeric value. We apply this to **all** race results (not just podium finishes) so that non-podium races contribute 0 to the running totals, keeping the cumulative counts accurate across a driver's full career.
- Three separate columns are created: `is_win` (finished 1st), `is_p2` (finished 2nd), `is_p3` (finished 3rd).

**Step 2 — Define cumulative window**

- `Window.partitionBy("driverId")` — scopes the window to one driver at a time, so cumulative counts are calculated independently for each driver.
- `.orderBy("raceId")` — orders races chronologically within each driver's partition. `raceId` increases monotonically with time (higher `raceId` = later race), so it serves as a reliable proxy for chronological order without needing the actual race date.
- `.rowsBetween(Window.unboundedPreceding, Window.currentRow)` — defines the window frame as "all rows from the very beginning of this driver's career up to and including the current row". This is what makes the sum **cumulative** — for any given race, it counts all podiums from race 1 through the current race.

**Steps 3 & 4 — Compute cumulative counts and format output**

- `F.sum("is_win").over(cum_win)` — computes the running total of wins for each driver up to and including each race. After race N, this equals the total number of times the driver finished 1st in races 1 through N. Same logic applies for `cum_p2` and `cum_p3`.
- `.join(races.select("raceId", "name", "year"), on="raceId", how="left")` — brings in the human-readable race name and season year.
- `.join(drivers.select("driverId", "forename", "surname"), on="driverId", how="left")` — brings in driver names to replace numeric `driverId`.
- `F.concat_ws(" ", "forename", "surname").alias("driver")` — combines first and last name into a single readable column.
- `.orderBy("year", "name", "positionOrder")` — sorts by season, then race name, then finishing position within each race, making it easy to read the podium standings race by race throughout the season.

# 6. [10 pts] Continue exploring the data by answering your own question.

**Question:** Which constructor (team) is the best finishing position per season, and how has this ranking shifted year-over-year?

**Logic:**

The `constructor_standings` table records the cumulative standings after every race — so the standings after the **last race of each season** represent the final championship result. My approach is:

1. Find the last round of each season by taking `max("round")` grouped by `year`.
2. Filter `constructor_standings` to keep only rows where `round == last_round`, giving us the final end-of-season standings for each year.
3. Join with `constructors` to get human-readable team names.
4. Filter to `final_position == 1` to show only the championship winner each year.


In [0]:
# Finding the last round:
last_round = (
    races.groupBy("year")
    .agg(F.max("round").alias("last_round"))
)

# Building q6_result:
q6_result = (
    constructor_standings
    .join(races.select("raceId", "year", "round"), on="raceId", how="left")
    .join(last_round, on="year", how="left")
    .filter(F.col("round") == F.col("last_round"))  # 只保留最后一站
    .join(
        constructors.select("constructorId", F.col("name").alias("constructor")),
        on="constructorId", how="left"
    )
    .select(
        "year",
        "constructor",
        F.col("points").cast(DoubleType()).alias("final_points"),
        F.col("position").cast(IntegerType()).alias("final_position"),
        F.col("wins").cast(IntegerType()).alias("total_wins"),
    )
    .filter(F.col("final_position") == 1)
    .orderBy("year", "final_position")
)

display(q6_result)

**Code Explanation:**

**Finding the last round:**
- `races.groupBy("year").agg(F.max("round").alias("last_round"))` — for each season, finds the highest `round` number, which corresponds to the final race of that season. For example, if a season has 20 races, `last_round=20`.

**Building q6_result:**
- `.join(races.select("raceId", "year", "round"), on="raceId", how="left")` — brings in `year` and `round` from the `races` table so we know which season and round each standings entry belongs to.
- `.join(last_round, on="year", how="left")` — brings in the `last_round` value for each season so we can compare it against each row's `round`.
- `.filter(F.col("round") == F.col("last_round"))` — keeps only the rows where the race is the final race of the season, giving us the end-of-season standings for each year.
- `.join(constructors.select(...), on="constructorId", how="left")` — replaces the numeric `constructorId` with the human-readable team name, aliased as `constructor`.
- `F.col("points").cast(DoubleType())` — explicitly casts `points` to double since the raw CSV stores it as a string due to `\N` values in some rows.
- `F.col("position").cast(IntegerType())` and `F.col("wins").cast(IntegerType())` — same reason, cast to integer for correct numeric handling.
- `.filter(F.col("final_position") == 1)` — keeps only the championship winner (position 1) for each year.
- `.orderBy("year", "final_position")` — sorts chronologically by season so we can read the championship history year by year.